# Module 3 - Scientific Data Manipulation
## 4. Outlier Detection

Outliers are unusually large or small values compared to the rest of the data. In scientific datasets, they may represent:

- measurement error,
- sensor malfunction,
- data-entry mistakes,
- rare but real events.

This means that detecting an outlier is only the first step. We must still decide whether the value should be removed, flagged, corrected, or kept.

### Learning goals

By the end of this notebook, you should be able to:

- recognise suspicious values,
- inspect data with simple summaries,
- apply z-score style logic,
- apply the IQR rule,
- create flags for unusual observations,
- choose a basic cleaning strategy.


## 4.1 Example Dataset

We create a small dataset with one suspicious temperature value.


In [1]:
import pandas as pd
import numpy as np


In [2]:
data = {
    "day": pd.date_range("2026-07-01", periods=10, freq="D"),
    "temperature": [24.1, 24.5, 24.3, 24.7, 25.0, 24.8, 24.6, 39.5, 24.9, 24.4]
}

df = pd.DataFrame(data)
df


,day,temperature
0,2026-07-01,24.1
1,2026-07-02,24.5
2,2026-07-03,24.3
3,2026-07-04,24.7
4,2026-07-05,25.0
5,2026-07-06,24.8
6,2026-07-07,24.6
7,2026-07-08,39.5
8,2026-07-09,24.9
9,2026-07-10,24.4


Even before formal calculations, the value `39.5` looks unusual compared with the rest of the temperatures.


## 4.2 Simple Inspection

A good first step is to inspect descriptive statistics and sorted values.


In [3]:
df["temperature"].describe()


count    10.000000
mean     26.080000
std       4.723417
min      24.100000
25%      24.425000
50%      24.650000
75%      24.875000
max      39.500000
Name: temperature, dtype: float64

In [4]:
df.sort_values(by="temperature", ascending=False)


,day,temperature
7,2026-07-08,39.5
4,2026-07-05,25.0
8,2026-07-09,24.9
5,2026-07-06,24.8
3,2026-07-04,24.7
6,2026-07-07,24.6
1,2026-07-02,24.5
9,2026-07-10,24.4
2,2026-07-03,24.3
0,2026-07-01,24.1


This often reveals suspicious values immediately. Simple inspection is important because it keeps the analysis interpretable.


## 4.3 Using Domain Knowledge

Not every unusual value is automatically wrong. In real research, domain knowledge matters.

For example:

- a room temperature of 39.5°C may be suspicious,
- a desert surface temperature of 39.5°C may be realistic,
- a negative humidity value would usually be physically impossible.

So, outlier detection should be treated as a decision-support tool rather than a fully automatic truth detector.


## 4.4 Z-score Style Detection

A simple idea is to measure how far a value is from the mean in units of standard deviation.

This is called a **z-score**. Values with a large absolute z-score are often treated as suspicious.


In [5]:
mean_temp = df["temperature"].mean()
std_temp = df["temperature"].std()

df["z_score"] = (df["temperature"] - mean_temp) / std_temp
df["outlier_z"] = df["z_score"].abs() > 2

df


,day,temperature,z_score,outlier_z
0,2026-07-01,24.1,-0.419188,False
1,2026-07-02,24.5,-0.334504,False
2,2026-07-03,24.3,-0.376846,False
3,2026-07-04,24.7,-0.292161,False
4,2026-07-05,25.0,-0.228648,False
5,2026-07-06,24.8,-0.270990,False
6,2026-07-07,24.6,-0.313332,False
7,2026-07-08,39.5,2.841164,True
8,2026-07-09,24.9,-0.249819,False
9,2026-07-10,24.4,-0.355675,False


A threshold of 2 is used here for demonstration. In practice, thresholds such as 2 or 3 are common, depending on the context.


## 4.5 IQR Method

Another common rule uses the **interquartile range (IQR)**. This method is often more robust because it depends on quartiles rather than the mean.


In [6]:
q1 = df["temperature"].quantile(0.25)
q3 = df["temperature"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df["outlier_iqr"] = (df["temperature"] < lower_bound) | (df["temperature"] > upper_bound)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

df


Q1: 24.424999999999997
Q3: 24.875
IQR: 0.45000000000000284
Lower bound: 23.749999999999993
Upper bound: 25.550000000000004


,day,temperature,z_score,outlier_z,outlier_iqr
0,2026-07-01,24.1,-0.419188,False,False
1,2026-07-02,24.5,-0.334504,False,False
2,2026-07-03,24.3,-0.376846,False,False
3,2026-07-04,24.7,-0.292161,False,False
4,2026-07-05,25.0,-0.228648,False,False
5,2026-07-06,24.8,-0.270990,False,False
6,2026-07-07,24.6,-0.313332,False,False
7,2026-07-08,39.5,2.841164,True,True
8,2026-07-09,24.9,-0.249819,False,False
9,2026-07-10,24.4,-0.355675,False,False


The IQR rule is widely used because it is less influenced by extreme values than mean-based methods.


## 4.6 Flagging Instead of Dropping Immediately

In many cases, the safest first step is not to delete an outlier immediately, but to create a flag column.


In [7]:
df["suspicious"] = df["outlier_z"] | df["outlier_iqr"]
df


,day,temperature,z_score,outlier_z,outlier_iqr,suspicious
0,2026-07-01,24.1,-0.419188,False,False,False
1,2026-07-02,24.5,-0.334504,False,False,False
2,2026-07-03,24.3,-0.376846,False,False,False
3,2026-07-04,24.7,-0.292161,False,False,False
4,2026-07-05,25.0,-0.228648,False,False,False
5,2026-07-06,24.8,-0.270990,False,False,False
6,2026-07-07,24.6,-0.313332,False,False,False
7,2026-07-08,39.5,2.841164,True,True,True
8,2026-07-09,24.9,-0.249819,False,False,False
9,2026-07-10,24.4,-0.355675,False,False,False


Flagging keeps the original information while still allowing us to inspect or exclude suspicious observations later.


## 4.7 Cleaning Strategies

Once an outlier is detected, some simple options are:

1. **Keep it** if it is a real event.
2. **Remove it** if it is clearly an error.
3. **Replace it with `NaN`** and handle it later as missing data.
4. **Cap it** to a reasonable threshold if that is justified by the research design.

Below we show two very simple examples.


In [8]:
removed = df[~df["suspicious"]]
replaced_with_nan = df.copy()
replaced_with_nan.loc[replaced_with_nan["suspicious"], "temperature"] = np.nan

print("Rows kept after removal:")
display(removed)

print("Rows where suspicious temperatures were replaced by NaN:")
display(replaced_with_nan)


Rows kept after removal:


,day,temperature,z_score,outlier_z,outlier_iqr,suspicious
0,2026-07-01,24.1,-0.419188,False,False,False
1,2026-07-02,24.5,-0.334504,False,False,False
2,2026-07-03,24.3,-0.376846,False,False,False
3,2026-07-04,24.7,-0.292161,False,False,False
4,2026-07-05,25.0,-0.228648,False,False,False
5,2026-07-06,24.8,-0.270990,False,False,False
6,2026-07-07,24.6,-0.313332,False,False,False
8,2026-07-09,24.9,-0.249819,False,False,False
9,2026-07-10,24.4,-0.355675,False,False,False


Rows where suspicious temperatures were replaced by NaN:


,day,temperature,z_score,outlier_z,outlier_iqr,suspicious
0,2026-07-01,24.1,-0.419188,False,False,False
1,2026-07-02,24.5,-0.334504,False,False,False
2,2026-07-03,24.3,-0.376846,False,False,False
3,2026-07-04,24.7,-0.292161,False,False,False
4,2026-07-05,25.0,-0.228648,False,False,False
5,2026-07-06,24.8,-0.270990,False,False,False
6,2026-07-07,24.6,-0.313332,False,False,False
7,2026-07-08,NaN,2.841164,True,True,True
8,2026-07-09,24.9,-0.249819,False,False,False
9,2026-07-10,24.4,-0.355675,False,False,False


Replacing suspicious values with `NaN` is often useful because it allows us to apply the missing-data techniques from the previous notebook.


## 4.8 A Second Example with Multiple Variables

Outlier checks can also be applied to other variables. Here we add a humidity column with one suspicious value.


In [9]:
df2 = pd.DataFrame({
    "day": pd.date_range("2026-07-01", periods=6, freq="D"),
    "temperature": [24.2, 24.4, 24.6, 24.5, 24.7, 24.6],
    "humidity": [55, 56, 54, 120, 57, 56]
})

df2


,day,temperature,humidity
0,2026-07-01,24.2,55
1,2026-07-02,24.4,56
2,2026-07-03,24.6,54
3,2026-07-04,24.5,120
4,2026-07-05,24.7,57
5,2026-07-06,24.6,56


In [10]:
df2["humidity_impossible"] = (df2["humidity"] < 0) | (df2["humidity"] > 100)
df2


,day,temperature,humidity,humidity_impossible
0,2026-07-01,24.2,55,False
1,2026-07-02,24.4,56,False
2,2026-07-03,24.6,54,False
3,2026-07-04,24.5,120,True
4,2026-07-05,24.7,57,False
5,2026-07-06,24.6,56,False


This example shows that domain-based rules can sometimes be even more informative than purely statistical ones.


## 4.9 Guided Practice

Complete the tasks below.

1. Compute the IQR of the temperature column.
2. Create lower and upper bounds using the 1.5 × IQR rule.
3. Flag suspicious temperatures.
4. Create a table with only the suspicious rows.


In [11]:
practice = df[["day", "temperature"]].copy()

q1 = ...
q3 = ...
iqr = ...
lower_bound = ...
upper_bound = ...

practice["outlier"] = ...
suspicious = ...

practice


,day,temperature,outlier
0,2026-07-01,24.1,Ellipsis
1,2026-07-02,24.5,Ellipsis
2,2026-07-03,24.3,Ellipsis
3,2026-07-04,24.7,Ellipsis
4,2026-07-05,25.0,Ellipsis
5,2026-07-06,24.8,Ellipsis
6,2026-07-07,24.6,Ellipsis
7,2026-07-08,39.5,Ellipsis
8,2026-07-09,24.9,Ellipsis
9,2026-07-10,24.4,Ellipsis


### Suggested solution


In [12]:
practice = df[["day", "temperature"]].copy()

q1 = practice["temperature"].quantile(0.25)
q3 = practice["temperature"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

practice["outlier"] = (practice["temperature"] < lower_bound) | (practice["temperature"] > upper_bound)
suspicious = practice[practice["outlier"]]

practice


,day,temperature,outlier
0,2026-07-01,24.1,False
1,2026-07-02,24.5,False
2,2026-07-03,24.3,False
3,2026-07-04,24.7,False
4,2026-07-05,25.0,False
5,2026-07-06,24.8,False
6,2026-07-07,24.6,False
7,2026-07-08,39.5,True
8,2026-07-09,24.9,False
9,2026-07-10,24.4,False


In [13]:
suspicious


,day,temperature,outlier
7,2026-07-08,39.5,True


## 4.10 Summary

In this notebook, you learned how to:

- identify suspicious values through simple inspection,
- think about outliers using domain knowledge,
- detect unusual observations with z-score style logic,
- detect unusual observations with the IQR rule,
- flag outliers instead of deleting them immediately,
- connect outlier handling with missing-data strategies.

In the next notebook, we focus on **time-based indexing**, which is essential for time series and sensor data alignment.
